In [0]:
visits = spark.table("workspace.default.air_visits").toPandas()
stores = spark.table("workspace.default.air_stores").toPandas()
dates = spark.table("workspace.default.date_info").toPandas()

In [0]:
print("visits shape:", visits.shape)
print("stores shape:", stores.shape)
print("dates shape:", dates.shape)

In [0]:
visits.info()

In [0]:
import pandas as pd
visits["visit_date"] = pd.to_datetime(visits["visit_date"])

In [0]:
visits.head()

In [0]:
print("From date: ", visits["visit_date"].min())
print("To date: ", visits["visit_date"].max())
print("Num of days: ", visits["visit_date"].max() - visits["visit_date"].min())

In [0]:
print("Num of unique stores: ", visits["air_store_id"].nunique())

In [0]:
import matplotlib.pyplot as plt

visits["visitors"].plot(kind="hist", bins=60)
plt.xlabel("visitors per store per day")
plt.title("distribution of daily visitor counts")
plt.show()

In [0]:
import numpy as np

np.log(visits["visitors"]).plot(kind="hist", bins=60)
plt.xlabel("visitors per store per day")
plt.title("distribution of daily visitor counts")
plt.show()

In [0]:
dates.info()

In [0]:
dates["calendar_date"] = pd.to_datetime(dates["calendar_date"])

In [0]:
dates.head()

In [0]:
visits_cal = visits.merge(dates, left_on="visit_date", right_on="calendar_date", how='left')
visits_cal.head()

In [0]:
visits_cal.isnull().sum()

In [0]:
dow = visits_cal.groupby("day_of_week")["visitors"].mean()
dow.sort_values(ascending=False)

In [0]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow = dow.reindex(order)

dow.plot(kind="bar")
plt.xlabel("day of week")
plt.ylabel("average visitors per day")
plt.title("average visitors by day of week")
plt.show()

In [0]:
holiday = visits_cal.groupby("holiday_flg")["visitors"].mean()
holiday

In [0]:
daily_total = visits.groupby("visit_date")["visitors"].sum()
daily_total.sort_values(ascending=False)

In [0]:
active_stores = visits.groupby("visit_date")["air_store_id"].nunique()
active_stores.sort_values(ascending=False)

In [0]:
fig, ax1 = plt.subplots()

ax1.plot(daily_total.index, daily_total.values, color="r")
ax1.set_ylabel("total visitors")

ax2 = ax1.twinx()
ax2.plot(active_stores.index, active_stores.values, color="g")
ax2.set_ylabel("active stores")

plt.title("daily visitor count vs active stores over time")
plt.show()

In [0]:
hist_len = visits.groupby("air_store_id")["visit_date"].count()
hist_len.describe()


In [0]:
hist_len.plot(kind="hist", bins=60)
plt.xlabel("days of history per store")
plt.ylabel("number of stores")
plt.title("distribution of history per store")
plt.show()

In [0]:
stores.info()

In [0]:
stores.head()

In [0]:
genre_counts = stores["air_genre_name"].value_counts()
genre_counts

In [0]:
genre_counts.plot(kind="bar")
plt.xlabel("genre")
plt.ylabel("number of stores")
plt.title("number of stores per genre")
plt.show()

In [0]:
stores["air_area_name"].value_counts()

In [0]:
stores["air_area_name"].nunique()

In [0]:
visits_stores = visits.merge(stores, left_on="air_store_id", right_on="air_store_id", how="left")
visits_stores.head()

In [0]:
visits_stores.groupby("air_genre_name")["visitors"].sum().sort_values(ascending=False)

In [0]:
visits_stores.groupby("air_area_name")["visitors"].sum().sort_values(ascending=False)

## Conclusions from the EDA

**The data**
- 252,108 visit records across 829 restaurants, running from 2016-01-01 to 2017-04-22 (about 16 months / 477 days).

**Demand is right-skewed**
- Daily visitor counts are heavily right-skewed: lots of quiet days and a long tail of busy ones.
- Taking `log(visitors)` makes the distribution look roughly normal, so it's probably worth modelling on a log scale. The skew also means predicting a *range* (P10–P90) will be more honest than a single number.

**Day of week is the strongest signal**
- Busiest: Saturday (26.3), Sunday (23.9), Friday (23.1). Quietest: Monday (17.2).
- Day-of-week clearly has to be a feature.

**Holidays bump demand up**
- Average visitors on holidays = 23.7 vs 20.8 on normal days (~14% higher), so `holiday_flg` is worth keeping.

**The upward "trend" is misleading**
- Total daily visitors rise over time — but the number of active stores rises in almost the exact same shape (from 48 stores on the first day to ~800 near the end).
- So the growth is mostly new restaurants being onboarded, not existing ones getting busier. Good reason to forecast **per store** rather than on the aggregate total.


**History per store is very uneven**
- Days of history per store: median 284, but ranges from just 20 up to 477.
- The short-history stores are a cold-start problem. This is an argument for a **global model** (one model learning across all stores) instead of a separate model per store.

**Cohorts for later**
- 14 genres and 103 areas. Izakaya (197 stores) and Cafe/Sweets (181) are both the most common genres and the highest in total visitors.
- Genre and area look like sensible groups to compare model performance across in the model-selection step.

**Decisions I'm taking into feature engineering**
1. Forecast per store, not on the aggregate.
2. Lean towards a global model to cope with short-history stores.
3. Core features: day-of-week, holiday flag, and lags / rolling averages of past visitors.
4. Log-transform the target, and produce P10–P90 ranges since demand is so spread out.